# SVC Processing Pipeline Demo

**Part 1** walks through the pipeline on a single spectrum — one step at a time — so each stage is easy to see.  
**Part 2** applies the same pipeline to a full folder of scans, filters out reference panels and outliers, then saves the results and plots the spectra.  
**Part 3** groups spectra into user-defined pairs (or any size groups) and averages them.

| Part | Content |
|---|---|
| 1 — Single spectrum | Load → inspect raw → process → visualize each step |
| 2 — Full folder | Load → filter references & outliers → process → save |
| 3 — Paired averages | Define index groups → average → plot individuals + means |

## Instrument Overview

The SVC HR-1024i is a field hyperspectral spectroradiometer that records target reflectance across the visible, near-infrared, and shortwave-infrared regions: approximately 400-700 nm, 700-1000 nm, and 1000-2500 nm, respectively.

| Detector array | Spectral region | Approximate range |
|---|---:|---:|
| Silicon | VNIR | 340-1012 nm |
| InGaAs | SWIR-1 | 972-1910 nm |
| Extended InGaAs | SWIR-2 | 1894-2517 nm |

Because these detector arrays overlap, each raw `.sig` file contains sequential sensor segments and small discontinuities that must be trimmed, matched, smoothed, and resampled into a continuous reflectance curve. The final pipeline uses 400-2500 nm to avoid the noisier UV/near-UV edge, where signal quality is reduced by low solar irradiance, detector sensitivity, and optical transmission.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[0]
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
for import_path in (PROJECT_ROOT, NOTEBOOKS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from pipeline.sig_processor import SigFileProcessor
from pipeline_demo.svc import (
    Spectrum,
    SpectraCollection,
    verify_demo_data,
    save_spectra_csv,
    average_pairs,
    plot_paired_averages,
)


In [ ]:
DEMO_DATA      = Path.cwd() / "pipeline_demo/demo_data"
RAW_SPECTRA_FOLDER = verify_demo_data(
    DEMO_DATA / "spectra",
    PROJECT_ROOT / "notebooks/pipeline_demo/demo_data_manifest.json",
)
OUTPUT_FOLDER  = PROJECT_ROOT / "pipeline_outputs/csv_exports"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

SPECTRA_FOLDER = OUTPUT_FOLDER / "demo_sig_processed"
SigFileProcessor(correction_type="bronze").process_sig_files(
    input_folder=str(RAW_SPECTRA_FOLDER),
    output_folder=str(SPECTRA_FOLDER),
    verbose=False,
)

print(f"Raw demo spectra      : {RAW_SPECTRA_FOLDER}")
print(f"Processed demo spectra: {SPECTRA_FOLDER}")
print(f"CSV outputs           : {OUTPUT_FOLDER}")


---
## Part 1 — Single spectrum

Loading one scan and stepping through the pipeline makes it easy to see what each stage does
before applying it to a full folder.

The summary printed below also serves as an **instrument check**:
- `sensor count = 3` confirms a three-array instrument (Si + InGaAs + extended InGaAs)
- `splice wavelengths` near 984 nm and 1 896 nm are the boundaries between those arrays
- `bands = 1024` confirms full-resolution mode

If any of those look unexpected, verify you are pointing at the right folder before processing the full dataset.

### Load

In [ ]:
single_spectrum = Spectrum(next(SPECTRA_FOLDER.glob("*.sig")))
print(single_spectrum)

### Raw

The SVC stores all three sensor arrays sequentially in one file.
When plotted in file order the wavelength axis **folds back twice** — once near 1 000 nm and once near 1 830 nm —
because each new sensor array starts at a lower wavelength than the previous one ended.

In [ ]:
single_spectrum.plot()

### Process

Four stages run in sequence:
1. **Trim** overlapping bands at each splice wavelength
2. **Splice correction** — multiplicative, linearly-varying correction that aligns sensor intensities across boundaries
3. **Gaussian smooth** — reduces high-frequency noise using a per-band bandwidth
4. **Gaussian resample** — outputs a standardised 400 – 2 500 nm grid at 1 nm spacing

In [ ]:
single_spectrum.process()
print(single_spectrum)

### Processing steps

Three panels show the spectrum at each stage.  Red dashed lines mark the splice wavelengths.

In [ ]:
single_spectrum.plot_processing_steps()

---
## Part 2 — Full folder

The same pipeline now runs on every `.sig` file in the folder.
Before processing, two filtering steps remove scans that should not be included in the analysis:

- **Reference scans** — the white Spectralon panel measured before each plot; reflectance is near 1.0 across the whole spectrum and must be excluded.
- **Outliers** — any scan whose mean reflectance falls more than 2.5 standard deviations from the group mean (e.g. obstructed view, instrument not settled).

### Load

In [ ]:
collection = SpectraCollection(SPECTRA_FOLDER)
print(collection)

The raw plot shows the sensor fold-backs for every scan.  Reference panel scans appear as nearly-flat lines at reflectance ≈ 1.0.

In [ ]:
collection.plot_raw()

In [ ]:
collection.filter_reference_scans()
collection.filter_outliers()

In [ ]:
collection.process()
print(collection)

Three-panel check on one scan to confirm the pipeline ran correctly.

In [ ]:
collection.plot_processing_steps(spectrum_index=0)

In [ ]:
collection.plot()

### Save

Written to a CSV under `OUTPUT_FOLDER`.  
Rows are individual scans; columns are wavelengths 400 – 2 500 nm (integer labels).

In [ ]:
spectra_path = save_spectra_csv(collection, OUTPUT_FOLDER / "spectra.csv")
print("Saved:", spectra_path)

---
## Part 3 — Paired averages

Two measurements are taken per leaf, so scans arrive in consecutive pairs.
`average_pairs()` groups spectra by explicit index tuples and returns one averaged
row per group.  `plot_paired_averages()` draws individual scans faded behind their
bold group mean — one colour per pair — so you can see both the within-leaf
agreement and the between-leaf variation at the same time.

### Define groups

Each tuple is a set of **0-based indices** into the collection's scan list.
Run `[s.name for s in collection.spectra]` to check the index-to-filename mapping,
then edit the tuples below to match your measurement design.

In [ ]:
# Check which index corresponds to which file:
for i, s in enumerate(collection.spectra):
    print(f"[{i}] {s.name}")

In [ ]:
groups = [
    tuple(range(i, min(i + 2, len(collection.spectra))))
    for i in range(0, len(collection.spectra), 2)
]

print("Using groups:")
for group_number, indices in enumerate(groups, start=1):
    names = ", ".join(collection.spectra[i].name for i in indices)
    print(f"  pair_{group_number}: indices {indices} -> {names}")

pairs = average_pairs(collection, groups=groups)
print(pairs)


In [ ]:
plot_paired_averages(collection, pairs, groups=groups)

### Save paired averages

One row per leaf (pair), columns are wavelengths 400 – 2 500 nm.

In [ ]:
pairs_path = OUTPUT_FOLDER / "spectra_paired.csv"
pairs.to_csv(pairs_path)
print("Saved:", pairs_path)